In [1]:
import gc
import random
import re
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, DepthAnythingForDepthEstimation, CLIPVisionModel
from tqdm.notebook import tqdm
import os
import shutil

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE        = 224
BATCH_SIZE      = 8
EPOCHS          = 30
LR              = 1e-4
NORMAL_MIN_NORM = 0.5

HYPERSIM_DIRS = [
    Path("/kaggle/input/datasets/poorviesadagopan/hypersim-geometry-1"),
    Path("/kaggle/input/datasets/poorviesadagopan/hypersim-geometry-2"),
    Path("/kaggle/input/datasets/poorviesadagopan/hypersim-geometry-3"),
    Path("/kaggle/input/datasets/poorviesadagopan/hypersim-geometry-4"),
]
HYPERSIM_FRAMES_PER_SCENE = 50

NORMAL_LAYER_CONFIGS = {
    "dinov2":        {"A": [12], "B": [3, 6, 9, 12], "C": [5, 7, 9, 11]},
    "clip":          {"A": [12], "B": [3, 6, 9, 12], "C": [5, 7, 9, 12]},
    "depth_anything":{"A": [12], "B": [3, 6, 9, 12], "C": [5, 7, 9, 11]},
    "vit_small":     {"A": [12], "B": [3, 6, 9, 12], "C": [6, 8, 10, 12]},
}

TARGET_MODEL = "vit_small"   # change per notebook

print(f"Device: {DEVICE}")
print(f"Target model: {TARGET_MODEL}")
print(f"Normal configs:")
for cond in ["A", "B", "C"]:
    print(f"  {cond}: {NORMAL_LAYER_CONFIGS[TARGET_MODEL][cond]}")

Device: cuda
Target model: vit_small
Normal configs:
  A: [12]
  B: [3, 6, 9, 12]
  C: [6, 8, 10, 12]


In [2]:
PARTWISE_SCENES_PER_CHUNK = 8

def read_hdf5(path):
    with h5py.File(path, "r") as f:
        key = next(iter(f.keys()))
        arr = np.asarray(f[key], dtype=np.float32)
    return torch.from_numpy(arr)

def count_invalid(rec):
    try:
        nc = torch.nan_to_num(read_hdf5(rec["normal_cam"]),   nan=0.0, posinf=0.0, neginf=0.0)
        nw = torch.nan_to_num(read_hdf5(rec["normal_world"]), nan=0.0, posinf=0.0, neginf=0.0)
        return (nc.norm(dim=-1) <= 0.5).sum().item() + (nw.norm(dim=-1) <= 0.5).sum().item()
    except Exception:
        return int(1e9)

def discover_hypersim_frames(roots, frames_per_scene=HYPERSIM_FRAMES_PER_SCENE, seed=SEED):
    records, seen_scenes = [], set()
    for root in roots:
        if not root.exists():
            print(f"WARNING: Skipping missing root: {root}")
            continue
        for scene_dir in sorted(root.iterdir()):
            if not scene_dir.is_dir() or scene_dir.name in seen_scenes:
                continue
            seen_scenes.add(scene_dir.name)
            images_dir = scene_dir / "images"
            if not images_dir.exists():
                continue
            scene_records = []
            for cam_final in sorted(images_dir.glob("scene_cam_*_final_hdf5")):
                m = re.search(r"scene_(cam_\d+)_final_hdf5", cam_final.name)
                if m is None:
                    continue
                cam_id   = m.group(1)
                cam_geom = images_dir / f"scene_{cam_id}_geometry_hdf5"
                if not cam_geom.exists():
                    continue
                for color_file in sorted(cam_final.glob("frame.*.color.hdf5")):
                    frame_id     = color_file.stem.replace(".color", "")
                    norm_cam_f   = cam_geom / f"{frame_id}.normal_bump_cam.hdf5"
                    norm_world_f = cam_geom / f"{frame_id}.normal_bump_world.hdf5"
                    semantic_f   = cam_geom / f"{frame_id}.semantic.hdf5"
                    if all(f.exists() for f in [norm_cam_f, norm_world_f, semantic_f]):
                        scene_records.append({
                            "source": "hypersim", "scene": scene_dir.name,
                            "image": color_file,
                            "normal_cam": norm_cam_f, "normal_world": norm_world_f,
                        })
            scene_records = sorted(scene_records, key=count_invalid)
            records.extend(scene_records[:frames_per_scene])
    rng = np.random.default_rng(seed)
    rng.shuffle(records)
    return records

def split_by_scene(records, val_fraction=0.2, seed=SEED):
    scenes = sorted(set(r["scene"] for r in records))
    rng    = np.random.default_rng(seed)
    rng.shuffle(scenes)
    n_val      = max(1, int(len(scenes) * val_fraction))
    val_scenes = set(scenes[:n_val])
    return [r for r in records if r["scene"] not in val_scenes], \
           [r for r in records if r["scene"] in val_scenes]

def chunk_by_scene(records, scenes_per_chunk=PARTWISE_SCENES_PER_CHUNK):
    scene_map = {}
    for r in records:
        scene_map.setdefault(r["scene"], []).append(r)
    scenes = sorted(scene_map)
    chunks = []
    for start in range(0, len(scenes), scenes_per_chunk):
        part = []
        for scene in scenes[start:start + scenes_per_chunk]:
            part.extend(scene_map[scene])
        if part:
            chunks.append(part)
    return chunks

class NormalDataset(Dataset):
    def __init__(self, records, train=False):
        self.records = records
        self.train   = train

    def __len__(self):
        return len(self.records)

    def _augment(self, image, normal_cam, normal_world):
        if random.random() < 0.5:
            image         = TF.hflip(image)
            normal_cam[0]   = -normal_cam[0]
            normal_world[0] = -normal_world[0]
            normal_cam   = TF.hflip(normal_cam)
            normal_world = TF.hflip(normal_world)
        if random.random() < 0.8:
            image = TF.adjust_brightness(image, 0.8 + 0.4 * random.random())
            image = TF.adjust_contrast(image,   0.8 + 0.4 * random.random())
            image = TF.adjust_saturation(image, 0.8 + 0.4 * random.random())
        return image, normal_cam, normal_world

    def __getitem__(self, idx):
        rec   = self.records[idx]
        image = read_hdf5(rec["image"]).permute(2, 0, 1).float()
        image = torch.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
        image = torch.clamp(image / (1.0 + image), 0.0, 1.0)

        def load_normal(key):
            n = read_hdf5(rec[key])
            n = torch.nan_to_num(n, nan=0.0, posinf=0.0, neginf=0.0)
            if n.dim() == 3 and n.shape[-1] == 3:
                n = n.permute(2, 0, 1)
            return n.float()

        normal_cam   = load_normal("normal_cam")
        normal_world = load_normal("normal_world")

        image        = TF.resize(image,        (IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True)
        normal_cam   = TF.resize(normal_cam,   (IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True)
        normal_world = TF.resize(normal_world, (IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True)

        if self.train:
            image, normal_cam, normal_world = self._augment(image, normal_cam, normal_world)

        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        nc_norm = normal_cam.norm(dim=0, keepdim=True).clamp_min(1e-6)
        nw_norm = normal_world.norm(dim=0, keepdim=True).clamp_min(1e-6)
        normal_cam   = normal_cam   / nc_norm
        normal_world = normal_world / nw_norm

        return {"image": image, "normal_cam": normal_cam, "normal_world": normal_world}


# ── Build loaders ──────────────────────────────────────────────────────────
hypersim_records              = discover_hypersim_frames(HYPERSIM_DIRS)
hypersim_train, hypersim_val  = split_by_scene(hypersim_records)
normal_train_parts            = chunk_by_scene(hypersim_train, PARTWISE_SCENES_PER_CHUNK) or [hypersim_train]
normal_val_loader             = DataLoader(NormalDataset(hypersim_val, train=False),
                                           batch_size=BATCH_SIZE, shuffle=False,
                                           num_workers=2, pin_memory=True)

print(f"Hypersim train: {len(hypersim_train)} | val: {len(hypersim_val)}")
print(f"Normal train chunks: {len(normal_train_parts)}")

Hypersim train: 1000 | val: 250
Normal train chunks: 3


In [3]:
def load_backbone(model_name):
    if model_name == "dinov2":
        model = AutoModel.from_pretrained(
            "facebook/dinov2-base", output_hidden_states=True)
        meta = {"patch_size": 14, "num_layers": 12, "hidden_dim": 768}

    elif model_name == "clip":
        model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch16")
        meta  = {"patch_size": 16, "num_layers": 12, "hidden_dim": 768}

    elif model_name == "vit_small":
        model = AutoModel.from_pretrained(
            "WinKawaks/vit-small-patch16-224", output_hidden_states=True)
        meta  = {"patch_size": 16, "num_layers": 12, "hidden_dim": 384}
        
    elif model_name == "depth_anything":
        full = DepthAnythingForDepthEstimation.from_pretrained(
            "depth-anything/Depth-Anything-V2-Base-hf"
        )
        model = full.backbone
        model.config.output_hidden_states = True
        meta = {"patch_size": 14, "num_layers": 12, "hidden_dim": 768}
        
    else:
        raise ValueError(f"Unknown model: {model_name}")

    meta["n_patches"] = IMG_SIZE // meta["patch_size"]
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    print(f"Loaded {model_name} | "
          f"dim={meta['hidden_dim']}  "
          f"patch={meta['patch_size']}  "
          f"n_patches={meta['n_patches']}")
    return model, meta

In [4]:
#Component 1: Readout projection (CLS → patches)
class ReadoutProject(nn.Module):
    """
    DPT 'project' readout: concatenate CLS token with each patch token,
    then project back to original dim.
    """
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)        
        self.proj = nn.Sequential(
            nn.Linear(2 * dim, dim),
            nn.GELU(),
        )

    def forward(self, tokens):
        # tokens: (B, 1+N, D) — position 0 is CLS
        tokens  = self.norm(tokens)           
        cls     = tokens[:, 0:1, :].expand(-1, tokens.shape[1] - 1, -1)
        patches = tokens[:, 1:, :]
        return self.proj(torch.cat([patches, cls], dim=-1))  # (B, N, D)


#Component 2: Reassemble block
class ReassembleBlock(nn.Module):
    """
    Tokens → spatial feature map at target resolution.
    factor=4  → ConvTranspose upsample 4x  (shallowest layer)
    factor=2  → ConvTranspose upsample 2x
    factor=1  → keep same resolution
    factor=0.5 → Conv stride-2 downsample  (deepest layer)

    The four factors are chosen so that after Reassemble, all four
    feature maps are at DIFFERENT resolutions that double each step
    (e.g. 7→14→28→56 for patch16 / 224px input).  This means that
    when FusionBlock adds skip + x they are ALWAYS the same spatial
    size — no interpolation of x is needed inside FusionBlock.
    """
    def __init__(self, in_dim, out_dim, n_patches, factor):
        super().__init__()
        self.n_patches = n_patches
        self.readout   = ReadoutProject(in_dim)
        self.proj      = nn.Conv2d(in_dim, out_dim, kernel_size=1)
        self.norm      = nn.BatchNorm2d(out_dim)    # <<< ADD THIS

        if factor == 4:
            self.resample = nn.ConvTranspose2d(out_dim, out_dim, kernel_size=4, stride=4)
        elif factor == 2:
            self.resample = nn.ConvTranspose2d(out_dim, out_dim, kernel_size=2, stride=2)
        elif factor == 1:
            self.resample = nn.Identity()
        elif factor == 0.5:
            self.resample = nn.Conv2d(out_dim, out_dim, kernel_size=3, stride=2, padding=1)
        else:
            raise ValueError(f"Unsupported factor: {factor}")

    def forward(self, tokens):
        x = self.readout(tokens)                              # (B, N, D)
        B, N, D = x.shape
        x = x.permute(0, 2, 1).reshape(B, D, self.n_patches, self.n_patches)
        x = self.proj(x)
        x = self.norm(x)
        x = self.resample(x)
        return x


#Component 3: ResidualConvUnit
class ResidualConvUnit(nn.Module):
    """Two conv3x3 with residual skip — the RefineNet building block."""
    def __init__(self, features):
        super().__init__()
        self.conv1 = nn.Conv2d(features, features, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(features, features, 3, padding=1, bias=False)
        self.act   = nn.GELU()

    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(self.act(x))))


#Component 4: FusionBlock
class FusionBlock(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.rcu_deep  = ResidualConvUnit(features)   
        self.rcu_fused = ResidualConvUnit(features)  

    def forward(self, skip, x=None):
        out = skip                                   
        if x is not None:
            out = out + self.rcu_deep(x)             
        out = self.rcu_fused(out)
        out = F.interpolate(out, scale_factor=2,
                            mode="bilinear", align_corners=True)
        return out




In [5]:
def normal_loss(pred, target):
    valid = target.norm(dim=1) > NORMAL_MIN_NORM
    if valid.sum() == 0:
        return (pred * 0).mean()
    cos_sim = (pred * target).sum(dim=1)
    return (1.0 - cos_sim[valid]).mean()

@torch.no_grad()
def compute_normal_metrics(pred, target):
    valid = target.norm(dim=1) > NORMAL_MIN_NORM
    if valid.sum() == 0:
        return {"mean_angle_err": float("nan"), "median_angle_err": float("nan")}
    cos_sim = (pred * target).sum(dim=1).clamp(-1, 1)
    angles  = torch.acos(cos_sim[valid]) * (180.0 / torch.pi)
    return {
        "mean_angle_err":   float(angles.mean()),
        "median_angle_err": float(angles.median()),
    }

In [6]:
class NormalDPTDecoder(nn.Module):
    """DPT decoder with 3-channel output + L2 normalization for surface normals."""
    FUSION_DIM = 256
    FACTORS    = [4, 2, 1, 0.5]

    def __init__(self, hidden_dim, n_patches, layer_indices):
        super().__init__()
        assert len(layer_indices) == 4
        self.layer_indices = layer_indices
        F_ = self.FUSION_DIM
        self.reassemble = nn.ModuleList([
            ReassembleBlock(hidden_dim, F_, n_patches, factor)
            for factor in self.FACTORS
        ])
        self.fusion = nn.ModuleList([FusionBlock(F_) for _ in range(4)])
        self.head   = nn.Sequential(
            nn.Conv2d(F_, F_ // 2, kernel_size=3, padding=1),
            nn.Upsample(size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=True),
            nn.Conv2d(F_ // 2, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 3, kernel_size=1),  
        )

    def forward(self, hidden_states):
        features = [
            self.reassemble[i](hidden_states[layer_idx])
            for i, layer_idx in enumerate(self.layer_indices)
        ]
        x = self.fusion[3](features[3])
        x = self.fusion[2](features[2], x)
        x = self.fusion[1](features[1], x)
        x = self.fusion[0](features[0], x)
        out = self.head(x)
        return F.normalize(out, dim=1)   # L2 normalize along channel dim


class NormalSingleLayerDecoder(nn.Module):
    """Single-layer decoder for normals (Condition A baseline)."""
    FUSION_DIM = 256

    def __init__(self, hidden_dim, n_patches, layer_index):
        super().__init__()
        self.layer_index = layer_index
        self.n_patches   = n_patches
        F_ = self.FUSION_DIM
        self.readout = ReadoutProject(hidden_dim)
        self.proj    = nn.Conv2d(hidden_dim, F_, kernel_size=1)
        self.up1     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.up2     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.up3     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.up4     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.head    = nn.Sequential(
            nn.Conv2d(F_, F_ // 2, 3, padding=1),
            nn.Upsample(size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=True),
            nn.Conv2d(F_ // 2, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 3, 1),
        )

    def forward(self, hidden_states):
        tokens = hidden_states[self.layer_index]
        x = self.readout(tokens)
        B, N, D = x.shape
        x = x.permute(0, 2, 1).reshape(B, D, self.n_patches, self.n_patches)
        x = self.proj(x)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        return F.normalize(self.head(x), dim=1)


def build_normal_decoder(condition, model_name, meta):
    layer_indices = NORMAL_LAYER_CONFIGS[model_name][condition]
    if condition == "A":
        decoder = NormalSingleLayerDecoder(
            hidden_dim  = meta["hidden_dim"],
            n_patches   = meta["n_patches"],
            layer_index = layer_indices[0],
        )
    else:
        decoder = NormalDPTDecoder(
            hidden_dim    = meta["hidden_dim"],
            n_patches     = meta["n_patches"],
            layer_indices = layer_indices,
        )
    n_params = sum(p.numel() for p in decoder.parameters())
    print(f"  Normal decoder params: {n_params/1e6:.2f}M")
    return decoder.to(DEVICE)

In [7]:
@torch.no_grad()
def get_clip_hidden_states(model, images):
    """
    model : CLIPVisionModel  (i.e. full.vision_model)
    images: [B, 3, 224, 224]
    Returns: tuple of 13 tensors  [B, 197, 768]  (embed + 12 layers)
    """
    if images.ndim == 3:
        images = images.unsqueeze(0)
    outputs = model(pixel_values=images, output_hidden_states=True)
    return outputs.hidden_states     

In [8]:
def get_hidden(backbone, model_name, images):
    if model_name == "clip":
        return get_clip_hidden_states(backbone, images)
    return backbone(pixel_values=images, output_hidden_states=True).hidden_states

def run_normal_experiment(model_name, condition, epochs=EPOCHS):
    print(f"  {model_name.upper()} | NORMAL | Condition {condition} "
          f"| layers={NORMAL_LAYER_CONFIGS[model_name][condition]}")
    print(f"{'='*60}")

    backbone, meta = load_backbone(model_name)
    decoder        = build_normal_decoder(condition, model_name, meta)

    optimizer = torch.optim.AdamW(decoder.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr           = LR,
        steps_per_epoch  = sum((len(p) + BATCH_SIZE - 1) // BATCH_SIZE for p in normal_train_parts),
        epochs           = epochs,
    )

    best_angle       = float("inf")
    best_state       = None
    history          = []
    nan_streak       = 0

    for epoch in range(epochs):
        decoder.train()
        train_loss, n_steps = 0.0, 0
        for part_idx, part_records in enumerate(normal_train_parts, start=1):
            part_loader = DataLoader(
                NormalDataset(part_records, train=True),
                batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
            for batch in tqdm(part_loader,
                              desc=f"E{epoch+1}/{epochs} normal p{part_idx}", leave=False):
                images       = batch["image"].to(DEVICE, non_blocking=True)
                normal_cam_gt = batch["normal_cam"].to(DEVICE, non_blocking=True)
                with torch.no_grad():
                    hidden = get_hidden(backbone, model_name, images)
                pred = decoder(hidden); del hidden
                loss = normal_loss(pred, normal_cam_gt)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
                optimizer.step(); scheduler.step()
                train_loss += loss.item(); n_steps += 1
        train_loss /= max(1, n_steps)

        decoder.eval()
        all_metrics = []
        with torch.no_grad():
            for batch in normal_val_loader:
                images        = batch["image"].to(DEVICE, non_blocking=True)
                normal_cam_gt = batch["normal_cam"].to(DEVICE, non_blocking=True)
                hidden        = get_hidden(backbone, model_name, images)
                pred          = decoder(hidden); del hidden
                all_metrics.append(compute_normal_metrics(pred, normal_cam_gt))

        mean_angle   = float(np.nanmean([m["mean_angle_err"]   for m in all_metrics]))
        median_angle = float(np.nanmean([m["median_angle_err"] for m in all_metrics]))
        history.append({"epoch": epoch+1, "train_loss": train_loss,
                        "mean_angle_err": mean_angle, "median_angle_err": median_angle})
        print(f"  Epoch {epoch+1:3d} | loss={train_loss:.4f}  "
              f"MeanAngle={mean_angle:.2f}°  MedianAngle={median_angle:.2f}°")

        if np.isfinite(mean_angle) and mean_angle < best_angle:
            best_angle = mean_angle
            best_state = {k: v.cpu().clone() for k, v in decoder.state_dict().items()}
            nan_streak = 0
            torch.save({"history": history, "best_angle": best_angle, "model_state": best_state},
                       f"/kaggle/working/phase2_{model_name}_normal_{condition}.pt")
        else:
            nan_streak = nan_streak + 1 if not np.isfinite(mean_angle) else 0

        if nan_streak >= 3:
            print(f"  STOPPING: 3 consecutive NaN epochs.")
            break

    if best_state is None:
        print("WARNING: No valid checkpoint — all metrics were NaN")
    else:
        decoder.load_state_dict(best_state)

    del backbone, decoder
    torch.cuda.empty_cache(); gc.collect()
    return history, best_angle

Sanity check

In [9]:
# def sanity_check(model_name):
#     print(f"Sanity check: {model_name}")
#     backbone, meta = load_backbone(model_name)
#     dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
#     with torch.no_grad():
#         hidden = get_hidden(backbone, model_name, dummy)

#     for condition in ["A", "B", "C"]:
#         n_dec = build_normal_decoder(condition, model_name, meta).eval()
#         with torch.no_grad():
#             out = n_dec(hidden)
#         print(f"  Normal {condition}: shape={tuple(out.shape)} min={out.min():.4f} "
#               f"max={out.max():.4f} nan={torch.isnan(out).any().item()}")
#         del n_dec

#     del backbone, hidden
#     torch.cuda.empty_cache()
#     print("Sanity check passed.\n")

# sanity_check(TARGET_MODEL)

Sanity check: vit_small


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded vit_small | dim=384  patch=16  n_patches=14
  Normal decoder params: 5.45M
  Normal A: shape=(2, 3, 224, 224) min=-0.5010 max=0.9850 nan=False
  Normal decoder params: 13.25M
  Normal B: shape=(2, 3, 224, 224) min=-0.8895 max=0.4795 nan=False
  Normal decoder params: 13.25M
  Normal C: shape=(2, 3, 224, 224) min=-0.8453 max=0.7141 nan=False
Sanity check passed.



In [ ]:
# def quick_verify(model_name, epochs=2):
#     print(f"QUICK VERIFY: {model_name} — {epochs} epochs each, all conditions")

#     results = {}

#     for condition in ["A", "B", "C"]:
#         try:
#             h, best = run_normal_experiment(model_name, condition, epochs=epochs)
#             last   = h[-1]
#             status = "OK" if np.isfinite(best) else "NaN"
#             results[f"normal_{condition}"] = status
#             print(f"  normal {condition}: {status} | Angle={best:.2f}° | "
#                   f"loss={last['train_loss']:.4f}")
#         except Exception as e:
#             results[f"normal_{condition}"] = "ERROR"
#             print(f"  normal {condition}: ERROR — {e}")

#     print(f"\nSummary for {model_name}:")
#     all_ok = all(v == "OK" for v in results.values())
#     for k, v in results.items():
#         print(f"  {k:20s}: {v}")
#     print(f"\n{'ALL CLEAR' if all_ok else 'ISSUES FOUND'} for {model_name}")
#     return all_ok

# for m in ["dinov2", "clip", "depth_anything", "vit_small"]:
#     quick_verify(m)

In [10]:
TARGET_MODELS = ["dinov2", "clip", "depth_anything", "vit_small"]

Update for both depth and normal

In [11]:
# BASE_PATH = "/kaggle/input/datasets/rcharan23/all-best"
# MODELS    = ["clip", "vit_small", "dinov2", "depth_anything"]

# print(f"\n{'='*65}")
# print(f"{'Model':<18} {'Cond':<6} {'MeanAngle°':>12} {'MedianAngle°':>14}")
# print(f"{'='*65}")

# for model_name in MODELS:
#     backbone, meta = load_backbone(model_name)

#     for cond in ["A", "B", "C"]:
#         src = f"{BASE_PATH}/phase2_{model_name}_normal_{cond}.pt"
#         if not os.path.exists(src):
#             print(f"  NOT FOUND: {src}")
#             continue

#         data    = torch.load(src, map_location=DEVICE)
#         decoder = build_normal_decoder(cond, model_name, meta)
#         decoder.load_state_dict(data["model_state"])
#         decoder.eval()

#         mean_angs, med_angs = [], []
#         with torch.no_grad():
#             for batch in normal_val_loader:
#                 images        = batch["image"].to(DEVICE, non_blocking=True)
#                 normal_cam_gt = batch["normal_cam"].to(DEVICE, non_blocking=True)
#                 hidden        = get_hidden(backbone, model_name, images)
#                 pred          = decoder(hidden)
#                 valid   = normal_cam_gt.norm(dim=1) > NORMAL_MIN_NORM
#                 if valid.sum() == 0:
#                     continue
#                 cos_sim = (pred * normal_cam_gt).sum(dim=1).clamp(-1, 1)
#                 angles  = torch.acos(cos_sim[valid]) * (180.0 / torch.pi)
#                 mean_angs.append(float(angles.mean()))
#                 med_angs.append(float(angles.median()))

#         mean_ang = float(np.nanmean(mean_angs))
#         med_ang  = float(np.nanmean(med_angs))
#         print(f"  {model_name:<18} {cond:<6} {mean_ang:>12.4f} {med_ang:>14.4f}")
#         del decoder

#     del backbone
#     torch.cuda.empty_cache()
#     gc.collect()

# print(f"{'='*65}")


Model              Cond     MeanAngle°   MedianAngle°


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias 

...

Loaded dinov2 | dim=768  patch=14  n_patches=16
  Normal decoder params: 6.43M
  dinov2             A           28.2521        23.2173
  Normal decoder params: 17.19M
  dinov2             B           24.0236        17.5445
  Normal decoder params: 17.19M
  dinov2             C           23.7986        17.1711


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loaded depth_anything | dim=768  patch=14  n_patches=16
  Normal decoder params: 6.43M
  depth_anything     A           23.0333        17.0844
  Normal decoder params: 17.19M
  depth_anything     B           23.0822        16.8554
  Normal decoder params: 17.19M
  depth_anything     C           21.4777        14.9330


In [ ]:
for model_name in ["dinov2", "clip", "depth_anything", "vit_small"]:
    for condition in ["A", "B", "C"]:
        history, best = run_normal_experiment(model_name, condition, epochs=EPOCHS)
        print(f"{model_name} normal {condition} done | MeanAngle = {best:.2f}°")

In [ ]:
print(f"Normal Results for {TARGET_MODEL}")
print(f"{'Cond':>6} | {'MeanAngle°':>12} | {'MedianAngle°':>14} | {'B→C Gain':>10}")

rows = {}
for cond in ["A", "B", "C"]:
    path = f"/kaggle/working/phase2_{TARGET_MODEL}_normal_{cond}.pt"
    if os.path.exists(path):
        data = torch.load(path, map_location="cpu")
        rows[cond] = data["best_angle"]   # mean angle saved at train time
    else:
        rows[cond] = float("nan")

gain = (rows["B"] - rows["C"]) / rows["B"] * 100 \
       if np.isfinite(rows["B"]) and np.isfinite(rows["C"]) else float("nan")

for cond in ["A", "B", "C"]:
    print(f"  {cond:>4} | {rows[cond]:>12.4f} | {'—':>14} | {'—':>10}")
print(f"  {'B→C':>4} | {'':>12} | {'':>14} | {gain:>9.2f}%")